In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration pour la visualisation
%matplotlib inline
sns.set_theme(style="whitegrid")

## Importation & Exploration

In [ ]:
df = pd.read_csv('bank-transactions.csv')

print("--- Structure inspection ---")
display(df.shape)

print("--- types ---")
display(df.dtypes)

print("--- head ---")
display(df.head())

print("--- describe ---")
display(df.describe())

In [ ]:
print("--- missing values percentage ---")
df.isna().mean() * 100

In [ ]:
missing = df.isna().mean() * 100
missing.sort_values().plot(kind="barh")
plt.title("Missing Values Percentage by Column")
plt.xlabel("Percentage (%)")
plt.show()

In [ ]:
df.duplicated(subset="transaction_id").sum()

In [ ]:
duplicates = df[df.duplicated(subset="transaction_id", keep=False)]
display(duplicates.sort_values("transaction_id").head(5))

## Nettoyage des données

Supprimer les doublons : conserver la première occurrence par transaction_id

In [ ]:
df = df.drop_duplicates(subset=["transaction_id"], keep="first")

df.duplicated(subset="transaction_id").sum()

Corriger les types de dates : unifier au format AAAA-MM-JJ HH:MM:SS

In [ ]:
df['date_transaction'] = pd.to_datetime(df['date_transaction'], format='mixed', dayfirst=True, errors='coerce')
df['date_transaction'] = df['date_transaction'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['date_transaction'].isna().sum()

In [ ]:
df['date_transaction'].head(5)

Corriger les montants : remplacer les virgules par des points, convertir en float

In [ ]:
if df['montant'].dtype == 'object':
    df['montant'] = df['montant'].str.replace(',', '.').astype(float)
df['montant'].isna().sum()

In [ ]:
df['montant'].dtype

Nettoyer la colonne solde_avant : supprimer le suffixe texte ' EUR', convertir en float

In [ ]:
# D. Nettoyer solde_avant (supprimer ' EUR')
df['solde_avant'] = (df['solde_avant'].astype(str).str.replace(' EUR', '', regex=False).str.replace(',', '.').astype(float))
df['solde_avant'].astype(str).str.contains('EUR').sum()

In [ ]:
df['solde_avant'].dtype

In [ ]:
df['solde_avant'].isna().sum()

Normaliser les devises : passer en majuscules (eur → EUR)

In [ ]:
df['devise'] = df['devise'].str.upper().str.strip()
df['devise'].unique()

Harmoniser segment_client : unifier la casse (PREMIUM → Premium)

In [ ]:
df['segment_client'] = df['segment_client'].str.strip().str.capitalize()
df['segment_client'].unique()

Supprimer les espaces parasites sur la colonne agence

In [ ]:
df['agence'] = df['agence'].str.strip()
df['agence'].isna().sum()


In [ ]:
df['agence'].unique()

Traiter les valeurs manquantes : imputation par médiane, mode .....

In [ ]:
df['segment_client'].isna().sum()

In [ ]:
df['segment_client'].mode()
df['segment_client'] = df['segment_client'].fillna(df['segment_client'].mode()[0])
df['segment_client'].isna().sum()

In [ ]:
df['agence'] = df['agence'].fillna('Inconnue')
df['agence'].isna().sum()

In [ ]:
df['score_credit_client'].isna().sum()

In [ ]:
df['score_credit_client'] = df['score_credit_client'].fillna(df['score_credit_client'].median())
df['score_credit_client'].isna().sum()

In [ ]:
print("Nettoyage terminé. Valeurs manquantes restantes :", df.isnull().sum().sum())

In [ ]:
df.isna().sum()

## Détection & Traitement des Valeurs Aberrantes

Appliquer IQR ou Z-score sur montant : identifier et flagguer les transactions hors bornes

In [ ]:
df["montant"].describe()

In [ ]:
Q1 = df['montant'].quantile(0.25)
Q3 = df['montant'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_montant = (df['montant'] < lower) | (df['montant'] > upper)

print(f"number of anomalies (IQR) in montant : {outliers_montant.sum()}")
print(f"Bornes IQR pour montant : {lower.round(2)} à {upper.round(2)}")


number of anomalies (IQR) in montant : 108
Bornes IQR pour montant : -5079.7 à 4758.24


2000

In [ ]:
#from scipy.stats import zscore
#df['z_score'] = zscore(df['montant'])
 
#outliers_z = df[(df['z_score'] > 3) | (df['z_score'] < -3)]
#len(outliers_z)

Appliquer IQR ou Z-score sur score credit client : détecter les scores négatifs ou > 850

In [118]:
df['score_credit_client'].describe()

count    2000.000000
mean      630.643000
std       125.246616
min      -100.000000
25%       591.000000
50%       645.000000
75%       696.000000
max      1500.000000
Name: score_credit_client, dtype: float64

In [117]:
df['score_credit_client'].skew()

np.float64(-0.5542064802578116)

In [123]:
from scipy import stats


# --- Étape 3.2 : Z-SCORE sur le SCORE CRÉDIT CLIENT ---
# Le Z-score mesure l'écart type par rapport à la moyenne. 
# On considère généralement comme anomalie un Z-score > 3 ou < -3.
z_scores = np.abs(stats.zscore(df['score_credit_client']))
outliers_score_z = (z_scores > 3)

# Ajout de la règle métier stricte (Scores négatifs ou > 850) en complément si nécessaire
outliers_score_metier = (df['score_credit_client'] < 0) | (df['score_credit_client'] > 850)

# Fusion des détections pour le score crédit
outliers_score_final = outliers_score_z | outliers_score_metier

print(f"number of anomalies (Z-Score > 3) in score_credit_client : {outliers_score_z.sum()}")
print(f"détecter les scores négatifs ou > 850 in score_credit_client : {outliers_score_metier.sum()}")

number of anomalies (Z-Score > 3) in score_credit_client : 3
détecter les scores négatifs ou > 850 in score_credit_client : 4


In [122]:
"""
Q1_s = df['score_credit_client'].quantile(0.25)
Q3_s = df['score_credit_client'].quantile(0.75)
IQR_s = Q3_s - Q1_s

lower = Q1_s - 1.5 * IQR_s
upper = Q3_s + 1.5 * IQR_s

outliers = df[(df['score_credit_client'] < lower) | (df['score_credit_client'] > upper)]
print(lower, upper)
len(outliers)
"""

"\nQ1_s = df['score_credit_client'].quantile(0.25)\nQ3_s = df['score_credit_client'].quantile(0.75)\nIQR_s = Q3_s - Q1_s\n\nlower = Q1_s - 1.5 * IQR_s\nupper = Q3_s + 1.5 * IQR_s\n\noutliers = df[(df['score_credit_client'] < lower) | (df['score_credit_client'] > upper)]\nprint(lower, upper)\nlen(outliers)\n"

In [ ]:
outliers['score_credit_client']

In [124]:
from scipy.stats import zscore
df['z_credit'] = zscore(df['score_credit_client'])
outliers_credit = df[(df['z_credit'] > 3) | (df["z_credit"] < -3 )]
len(outliers_credit)

3

In [ ]:
df.drop(columns=['z_credit'], inplace=True)

In [ ]:
df['score_invalid'] = ((df['score_credit_client'] < 0) | (df['score_credit_client'] > 850))
df['score_invalid'].sum()

In [ ]:
df[df['score_invalid'] == True]['score_credit_client']

Créer une colonne is_anomaly (booléen) pour marquer les lignes aberrantes

In [ ]:
df['is_anomaly'] = df['montant_outlier'] | df['score_invalid']
df['is_anomaly'].sum()

In [ ]:
df['score_invalid'].sum()

Documenter la décision retenue : suppression, correction ou conservation avec flag

- No suppression applied
- No correction applied

- All anomalies were conservedand flagged using:
    -'montant_outlier' for transaction amounts
    -'score_invalid' for invalid credit scores
    -'is_anomaly' as a global indicator

Preserves all data while flagging anomalies for analysis, Helps detecting anomalies without losing valuable data.

## Feature Engineering

Extraire depuis date_transaction : annee, mois, trimestre, jour-semaine

In [ ]:
df['date_transaction'] = pd.to_datetime(df['date_transaction'], errors='coerce')

In [ ]:
df['annee'] = df['date_transaction'].dt.year
df['mois'] =df['date_transaction'].dt.month
df['trimestre'] = df['date_transaction'].dt.quarter
df['jour_semaine'] = df['date_transaction'].dt.day_name()

Calculer montant_eur_verifie = montant / taux_change_eur et comparer à montant_eur existant

In [ ]:
# Recalcul
df['montant_eur_verifie'] = df['montant'] / df['taux_change_eur']

# Différence
df['difference_eur'] = df['montant_eur'] - df['montant_eur_verifie']

In [ ]:
df['difference_eur'].abs().sum()

In [ ]:
df[df['difference_eur'].abs() > 1][
    ['devise', 'montant', 'taux_change_eur', 'montant_eur', 'montant_eur_verifie']
]

Créer categorie_risque basée sur score_credit_client : Low (>= 700), Medium (580–699), High (< 580)

In [ ]:
def classify_risk(score):
    if score < 580:
        return 'High'
    elif score <700:
        return 'Medium'
    else:
        return 'Low'

df['categorie_risque'] = df['score_credit_client'].apply(classify_risk)
df[['score_credit_client', 'categorie_risque']].head()

Calculer solde_net par client : total crédits – total débits

In [ ]:
solde_client = df.groupby('client_id')['montant'].sum().to_frame(name='solde_net')

solde_client.head()

In [ ]:
df['type_operation'].unique()

In [ ]:
# Separate credit & debit
df['montant_credit'] = df['montant'].where(df['type_operation'] == 'Credit', 0)
df['montant_debit'] = df['montant'].where(df['type_operation'] == 'Debit', 0)

In [ ]:
# Group by client
solde_client = df.groupby('client_id')[['montant_credit', 'montant_debit']].sum()
# Calculate solde
solde_client['solde_net'] = solde_client['montant_credit'] + solde_client['montant_debit']

In [ ]:
solde_client.head()

Agréger par client : nb_transactions, montant_moyen, nb_produits_distincts

In [127]:
agg_client = df.groupby('client_id').agg(
    nb_transactions = ('transaction_id', 'count'),
    montant_moyen = ('montant', 'mean'),
    nb_produits_distincts = ('categorie', 'nunique')
)
agg_client.head(6)

,nb_transactions,montant_moyen,nb_produits_distincts
client_id,,,
CLI0001,14,108.324286,6
CLI0002,12,-403.411667,6
CLI0003,13,-777.826923,6
CLI0004,16,-94.131250,7
CLI0005,9,-708.276667,5
CLI0006,16,-157.873750,8


Créer taux_rejet : proportion de transactions rejetées par agence

In [ ]:
df['statut'].unique()

In [ ]:
df['is_rejected'] = df['statut'] == 'Rejete'

taux_rejet = df.groupby('agence')['is_rejected'].mean().to_frame(name='taux_rejet')

taux_rejet['taux_rejet'] = (taux_rejet['taux_rejet'] * 100).round(2)

taux_rejet.head()

In [ ]:
"""
A. Extraire les composants de la date
df['annee'] = df['date_transaction'].dt.year
df['mois'] = df['date_transaction'].dt.month
df['trimestre'] = df['date_transaction'].dt.quarter
df['jour_semaine'] = df['date_transaction'].dt.day_name()

# B. Calculer montant_eur_verifie
df['montant_eur_verifie'] = df['montant'] / df['taux_change_eur']

# C. Créer categorie_risque
def risk_mapping(score):
    if score >= 700: return 'Low'
    elif 580 <= score <= 699: return 'Medium'
    else: return 'High'

df['categorie_risque'] = df['score_credit_client'].apply(risk_mapping)

# D. Agréger par client (Solde net, nb transactions, etc.)
client_metrics = df.groupby('client_id').agg(
    nb_transactions=('transaction_id', 'count'),
    montant_moyen=('montant', 'mean'),
    nb_produits_distincts=('produit', 'nunique')
).reset_index()

# Fusion avec le dataset principal pour enrichir
df = df.merge(client_metrics, on='client_id', how='left')

# E. Taux de rejet par agence
rejet_agence = df[df['statut'] == 'Rejete'].groupby('agence').size() / df.groupby('agence').size()
df['taux_rejet_agence'] = df['agence'].map(rejet_agence).fillna(0)
"""

## Export

Cleaning and Organizing Columns

In [ ]:
# Drop helper / unnecessary columns
df.drop(columns=[
    'montant_credit',
    'montant_debit',
    'is_rejected',
    'difference_eur',
    'taux_interet',
    'montant_outlier',
    'score_invalid'
], inplace=True, errors='ignore')

In [ ]:
# Reorder columns (clean logical structure)
columns_order = [
    # Identifiers
    'transaction_id', 'client_id',
    # Date
    'date_transaction', 'annee', 'mois', 'trimestre', 'jour_semaine',
    # Transaction info
    'categorie', 'produit', 'type_operation', 'statut', 'agence',
    # Financial
    'montant', 'devise', 'taux_change_eur',
    'montant_eur', 'montant_eur_verifie',
    # Client info
    'score_credit_client', 'categorie_risque',
    'segment_client', 'solde_avant',
    # Aggregations
    'nb_transactions', 'montant_moyen', 'nb_produits_distincts',
    # Flags
    'is_anomaly',
    
]

# Apply order safely
df = df[[col for col in columns_order if col in df.columns]]


Vérifier la cohérence globale du dataset final (aucune valeur manquante critique, types corrects)

In [ ]:
print("Missing values per column: ")
print(df.isnull().sum())

print("\n Data type and info: ")
df.info()

Exporter le dataset propre et enrichi : financecore_clean.csv

In [ ]:
df = df.round(2)
df.to_csv("financecore_clean.csv", index=False)

Documenter chaque décision de nettoyage dans un fichier DECISIONS.md